# Notes
Auto read `data_input.json` and then predict IC20s and hazard levels for them.

The default output will be written to `<working dir>/data_output.csv`

In [ ]:
# Public methods
import os, json, tqdm

import numpy as np
import pandas as pd

def join(*path) -> str:
    return os.path.join(WORKING_DIR, *path)

WORKING_DIR = "Viability"


# Predict
Set up running device at constant `DEVICE` before running.

For models, we have **binary classification models** (indicate $\mathrm{active}$ or $\mathrm{inactive}$) and **regression models** (predict $IC_\mathrm{20,free}$ and $IC_\mathrm{20,cell}$)  
Each type of model has 3 endpoints (**viability**/**apoptosis**/**mitochondrial toxicity**), you can comment out the endpoints if not need

In [ ]:
# Define running parameters
from Estimator.estimators import ViabilityEstimator

# define running device
DEVICE = "cuda:1"

# define endpoints
ENDPOINTS = [
    'Hazard_viability',
    'IC20_free_viability',
    'IC20_cell_viability',
    'Hazard_mitochondrial',
    'IC20_free_mitochondrial',
    'IC20_cell_mitochondrial',
    'Hazard_apoptosis',
    'IC20_free_apoptosis',
    'IC20_cell_apoptosis',
]

# load config
with open("config.json", "r") as f:
    config = json.load(f)

# input data
input_data = pd.read_json(join("data_input.json"))

# set endpoints
estimator_list = [
    ViabilityEstimator(device=DEVICE, **config[name]) for name in ENDPOINTS
]


In [ ]:
# Predict

# init
shape = (len(estimator_list), input_data.shape[0])
result_array = np.zeros(shape, dtype=np.float64)

for idx in tqdm.trange(len(estimator_list)):
    estimator = estimator_list[idx]
    result_array[idx] = estimator.predict_auto(input_data)


In [ ]:
# Results

# dataframe
result_df = pd.DataFrame(result_array.T, columns=[i.date for i in estimator_list])

# rename columns for better readability
model_mappings = {config[name]['date']:name for name in config.keys()}
result_df = result_df.rename(columns=model_mappings)

# merge
smiles_series = input_data['smiles']
result_df = pd.concat(
    [smiles_series, result_df],
    axis=1,
)

In [ ]:
# export results
result_df.to_csv(join("data_output.csv"), index=False)